# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides an interactive guide for loading and exploring a clinical oncology dataset using the `mlcroissant` library. All references to the dataset's structural components (record sets, fields) are made using their Croissant `@id` identifiers, ensuring reproducibility and compliance with the schema.

### Dataset Source
The dataset is described using a Croissant JSON-LD schema:
**[Croissant schema URL](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)**

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review the available record sets, their field schemas, and all Croissant `@id` values for future referencing.

Below, we print summaries for each record set found in the dataset.

In [ ]:
# List all record sets and their fields by @id
record_sets = list(dataset.record_sets)
print(f"Number of record sets: {len(record_sets)}\n")
record_set_ids = []
for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']}")
    print(f"  Name: {rs.get('name', '-')}")
    print(f"  Description: {rs.get('description', '-')}")
    print("  Fields:")
    for field in rs.get('field', []):
        print(f"    - @id: {field['@id']}, name: {field.get('name', '-')}, dataType: {field.get('dataType', '-')}" )
    print()
    record_set_ids.append(rs['@id'])

# For quick overview, show all record set ids we can use later
print('RecordSet @ids discovered:', record_set_ids)

## 3. Data Extraction
Load data from selected record sets into Pandas DataFrames for analysis. You should always use the `@id` of each record set and field.

In [ ]:
dataframes = dict()

# Replace with the actual record set @ids discovered above. Here we just use the discovered record_sets list
for rs_id in record_set_ids:
    print(f"Loading data for RecordSet: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"- Columns (@id): {df.columns.tolist() if not df.empty else 'No records found.'}")
    if not df.empty:
        display(df.head(3))

# For demonstration, pick first record set with data to proceed
main_record_set_id = None
for rs_id in record_set_ids:
    if not dataframes[rs_id].empty:
        main_record_set_id = rs_id
        break
if main_record_set_id is not None:
    print(f"\nProceeding with main_record_set_id: {main_record_set_id}")
else:
    raise RuntimeError("No record sets with data found!")

## 4. Exploratory Data Analysis (EDA)
Apply basic data processing steps: filtering numeric values, normalization, and grouping. All columns are referenced by their `@id` as defined in the Croissant schema record set above.

In [ ]:
# Identify a numeric field @id from the main record set for demo purposes
df = dataframes[main_record_set_id]
numeric_field_id = None
possible_numeric_types = {'Integer', 'Float', 'Number'}

# Find a numeric column in the main record set
for rs in dataset.record_sets:
    if rs['@id'] == main_record_set_id:
        for field in rs.get('field', []):
            if field.get('dataType','').split(':')[-1] in possible_numeric_types and field['@id'] in df.columns:
                numeric_field_id = field['@id']
                break

if numeric_field_id is None:
    print("No numeric field found in the main record set. Skipping numeric filtering and normalization.")
else:
    print(f"Using numeric field: {numeric_field_id}\n")
    # Convert to numeric (in case it was loaded as string)
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].quantile(0.8)  # use 80th percentile as sample threshold
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (top 20%)")
    display(filtered_df.head())

    # Normalization
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    # Try grouping by a categorical field present
    group_field_candidates = []
    for rs in dataset.record_sets:
        if rs['@id'] == main_record_set_id:
            group_field_candidates = [f['@id'] for f in rs.get('field', []) if f.get('dataType','').split(':')[-1] in ['Text', 'String'] and f['@id'] in df.columns]
            break
    if group_field_candidates:
        group_field_id = group_field_candidates[0]
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped average of {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())
    else:
        print("No text/String groupable field found in the main record set.")

## 5. Visualization
Visualize distributions or relationships between data fields. Below we plot the numeric field (if found) and its grouped means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

if numeric_field_id is not None:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if 'grouped_df' in locals():
        plt.figure(figsize=(10,4))
        sns.barplot(data=grouped_df, x=grouped_df.columns[0], y=grouped_df.columns[1])
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45, ha='right')
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we loaded and explored a clinical oncology dataset package using the `mlcroissant` library. By referencing entities by their Croissant `@id`, we programmatically accessed record sets and fields for analysis and visualization. You can customize this workflow for deeper exploration, modeling, or further EDA as needed for your ML/AI or clinical informatics research.

<small>Notebook auto-generated for FAIR Clinical Data Exploration with mlcroissant (2024)</small>